In [59]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["ANTHROPIC_API_KEY"]=os.getenv("ANTHROPIC_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [60]:
from langsmith import Client
client = Client()

dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['636c468f-a61a-4db3-8b90-8522dfae585b',
  'bae3b633-2dc2-4184-81a3-507fd68c0893',
  'b9e3ffbc-a370-46c9-a48f-4ad5422fa20e',
  '32107412-f39e-4653-9157-997594c8c9e4',
  'bf963ac9-f8dc-4524-b108-48abed893419'],
 'count': 5,
 'as_of': '2026-06-08T13:11:18.986772675Z'}

In [61]:
import os
from groq import Groq
from langsmith import traceable

groq_client = Groq()

eval_instructions = "You are an expert proffessor specialized in grading students' answers to questions."

def correctness(outputs:dict, reference_answer:dict)-> bool:
    # Extract the string values from your model output and ground truth
    ai_answer = outputs.get("output") or outputs.get("result")
    ground_truth = reference_outputs.get("output") or reference_outputs.get("answer")
    
    # Simple evaluation logic (e.g., direct match or string inclusion)
    is_correct = ground_truth.lower() in ai_answer.lower()
    
    return {
        "key": "correctness",
        "score": 1.0 if is_correct else 0.0
    }

In [62]:
def concision(outputs: dict) -> dict:
    ai_answer = outputs.get("output") or outputs.get("result")
    
    # Score 1.0 if short, 0.0 if too verbose
    is_concise = len(ai_answer.split()) < 50 
    
    return {
        "key": "concision",
        "score": 1.0 if is_concise else 0.0
    }

In [63]:
from langsmith.schemas import Run, Example

def correctness(run: Run, example: Example) -> dict:
    # System output lives inside run.outputs
    # Ground truth lives inside example.outputs
    score = run.outputs["output"] == example.outputs["answer"]
    return {"key": "correctness", "score": score}

Run Evaluations

In [64]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

@traceable(run_type="llm")
def my_app(question: str, model: str = "llama-3.1-8b-instant", instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens = 1000,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [65]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"output": my_app(inputs["question"])}

In [66]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="groq-chatbot"
)

View the evaluation results for experiment: 'groq-chatbot-9f608aab' at:
https://smith.langchain.com/o/a6f26838-ef95-51cc-b455-90de5bbfde12/datasets/57c47f78-e7b9-4d5f-90f8-70bb1e96d3cb/compare?selectedSessions=9e083605-52bd-4c02-82bb-02ef2e4c30c9




5it [00:01,  2.54it/s]
